# Phase 3: 분석 쿼리
**DB:** `data/franchise.db` (맛나국밥 가상 프랜차이즈)

| 단계 | 내용 |
|------|------|
| 1 | HAVING — 집계 결과 필터링 |
| 2 | 서브쿼리 — SELECT/HAVING 안에 쿼리 중첩 |
| 3 | 윈도우 함수 — RANK() |

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/franchise.db')

def query(sql):
    return pd.read_sql_query(sql, conn)

---
## 1. HAVING
### 과제 3-1
`daily_sales` × `stores` JOIN 후 **연간 총매출이 5억 5천만 원 이상인 매장**만 조회  
> `WHERE`는 행 단위 필터, `HAVING`은 **집계 결과** 필터  
> 실행 순서: `FROM` → `JOIN` → `GROUP BY` → `HAVING` → `ORDER BY`

In [2]:
query("""
SELECT store_name, SUM(total_amount) AS 총매출
FROM daily_sales
JOIN stores ON daily_sales.store_id = stores.store_id
GROUP BY store_name
HAVING SUM(total_amount) >= 550000000
ORDER BY 총매출 DESC
""")

,store_name,총매출
0,맛나국밥 영등포구점,654547000
1,맛나국밥 용인시점,629635000
2,맛나국밥 고양시점,620811000
3,맛나국밥 강남구점,604537000
4,맛나국밥 달서구점,601874000
5,맛나국밥 강서구점,589305000
6,맛나국밥 노원구점,580658000
7,맛나국밥 종로구점,557547000


### 과제 3-2
`daily_sales` × `menu_items` JOIN 후 **카테고리별 평균 판매수량이 15 이상인 카테고리**만 조회  
> 평균은 `AVG(quantity)` — `AVG(total_amount)`는 금액이라 의미 다름

In [3]:
query("""
SELECT category, AVG(quantity) AS 평균수량
FROM daily_sales
JOIN menu_items ON daily_sales.menu_id = menu_items.menu_id
GROUP BY category
HAVING AVG(quantity) >= 15
ORDER BY 평균수량 DESC
""")

,category,평균수량
0,탕,23.950913
1,사이드,20.780685


---
## 2. 서브쿼리
### 과제 3-3
**전체 평균 매출보다 높은 매장**만 조회  
> 단계별 구조:
> 1. 매장별 총매출 구하기 (내부 서브쿼리)
> 2. 그 결과의 AVG 구하기 (외부 서브쿼리)
> 3. HAVING 조건에 서브쿼리 삽입

In [4]:
query("""
SELECT store_name, SUM(total_amount) AS 총매출
FROM daily_sales
JOIN stores ON daily_sales.store_id = stores.store_id
GROUP BY store_name
HAVING 총매출 >= (
    SELECT AVG(총매출)
    FROM (
        SELECT store_name, SUM(total_amount) AS 총매출
        FROM daily_sales
        JOIN stores ON daily_sales.store_id = stores.store_id
        GROUP BY store_name
    ) AS sub
)
ORDER BY 총매출 DESC
""")

,store_name,총매출
0,맛나국밥 영등포구점,654547000
1,맛나국밥 용인시점,629635000
2,맛나국밥 고양시점,620811000
3,맛나국밥 강남구점,604537000
4,맛나국밥 달서구점,601874000
5,맛나국밥 강서구점,589305000
6,맛나국밥 노원구점,580658000
7,맛나국밥 종로구점,557547000
8,맛나국밥 해운대구점,535210000


---
## 3. 윈도우 함수
### 과제 3-4
매장별 총매출에 **순위** 부여 (`RANK()`)  
> `RANK() OVER (ORDER BY ...)` — GROUP BY처럼 집계하면서 행을 유지하고 각 행에 순위를 붙임  
> 동점이면 같은 순위, 다음 순위는 건너뜀 (1,2,2,4 ...)

In [5]:
query("""
SELECT
    store_name,
    SUM(total_amount) AS 총매출,
    RANK() OVER (ORDER BY SUM(total_amount) DESC) AS 순위
FROM daily_sales
JOIN stores ON daily_sales.store_id = stores.store_id
GROUP BY store_name
ORDER BY 총매출 DESC
""")

,store_name,총매출,순위
0,맛나국밥 영등포구점,654547000,1
1,맛나국밥 용인시점,629635000,2
2,맛나국밥 고양시점,620811000,3
3,맛나국밥 강남구점,604537000,4
4,맛나국밥 달서구점,601874000,5
5,맛나국밥 강서구점,589305000,6
6,맛나국밥 노원구점,580658000,7
7,맛나국밥 종로구점,557547000,8
8,맛나국밥 해운대구점,535210000,9
9,맛나국밥 서초구점,521674000,10


In [6]:
conn.close()